# 06 — Classification

Mục tiêu: dự đoán `parking_behavior` từ các feature đã được tạo ở `05_feature_engineering.ipynb`.

Các lớp: `Early_Return`, `Full_Day`, `Overnight`, `Long_Term`.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

data_path = '../data/processed/classification_data.csv'
df = pd.read_csv(data_path)

print("Kích thước dữ liệu:", df.shape)
display(df.head())

In [ ]:
# Tách Features và Target
X = df.drop(columns=['parking_behavior'])
y = df['parking_behavior']

print("Số lượng feature:", X.shape[1])
print("Features:")
print(X.columns.tolist())

print("
Target:")
print(y.name)

print("
Phân bố target:")
print(y.value_counts())

print("
Tỷ lệ target (%):")
print((y.value_counts(normalize=True) * 100).round(2))

In [ ]:
# Chia train/test theo tỷ lệ 80/20 và giữ nguyên tỷ lệ các lớp
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
# Khai báo các model cần so sánh.
# Class weight giúp mô hình chú ý hơn tới các lớp ít mẫu.

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"Đang huấn luyện: {name}")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'model': name,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })
    
    trained_models[name] = model
    
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print("-" * 50)

results_df = (
    pd.DataFrame(results)
      .sort_values(
          by=['macro_f1', 'accuracy'],
          ascending=False
      )
      .reset_index(drop=True)
)

print("
Bảng so sánh:")
display(results_df.round(4))

In [ ]:
# Chọn model tốt nhất theo Macro F1
best_model_name = results_df.loc[0, 'model']
best_classification_model = trained_models[best_model_name]

y_pred_best = best_classification_model.predict(X_test)

print("Model tốt nhất:", best_model_name)
print("Accuracy:", round(
    accuracy_score(y_test, y_pred_best), 4
))
print("Macro F1:", round(
    f1_score(y_test, y_pred_best, average='macro'), 4
))

print("
Classification Report:")
print(
    classification_report(
        y_test,
        y_pred_best,
        zero_division=0
    )
)

print("
Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

In [ ]:
# Lưu model và kết quả so sánh
os.makedirs('../models/classification', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

model_path = '../models/classification/best_classification_model.pkl'
results_path = '../data/processed/classification_model_results.csv'

joblib.dump(best_classification_model, model_path)
results_df.to_csv(results_path, index=False)

print("Đã lưu model tốt nhất:")
print(model_path)

print("
Đã lưu kết quả so sánh:")
print(results_path)

print("
Thứ tự feature của model:")
print(best_classification_model.feature_names_in_.tolist())

In [ ]:
# Kiểm tra model sau khi lưu bằng một mẫu test thực tế
sample = X_test.iloc[[0]]
sample_prediction = best_classification_model.predict(sample)[0]

print("Feature mẫu:")
display(sample)

print("Dự đoán parking_behavior:", sample_prediction)
print("Thực tế:", y_test.iloc[0])